# 03 — Forecasting Models for Youth Unemployment

**Project:** Forecasting Youth Unemployment in Pakistan
**Author:** Awais Ahmed Shah — BS Data Science (Final Year), UET Peshawar

## Goal
Can simple, interpretable models forecast Pakistan's youth unemployment rate?
We compare five approaches on a **time-based hold-out test** (train ≤ 2015, test 2016–2025)
using a consistent **one-step-ahead** evaluation:

| Model | Idea |
|---|---|
| Naive | Next year = this year's value |
| Drift | Next year = this year + average yearly change seen in training |
| ARIMA | Classical time-series model (order chosen by AIC on the training data) |
| Linear Regression | Predicts from lag features (last 1–3 years + 3-year rolling mean) |
| Random Forest | Same lag features, non-linear tree ensemble |

One-step-ahead means every test prediction uses the *observed* values up to the
previous year. This is the fair way to compare models on a short annual series,
but it is **easier** than true multi-year forecasting — the final 2026–2030
outlook at the end is therefore presented with explicit uncertainty warnings.

Metrics (MAE and RMSE, in percentage points) are written to `results/model_metrics.csv`.

In [1]:
from pathlib import Path

def find_project_root():
    """Walk up from the notebook folder until the folder containing data/ is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (folder containing data/processed).")

ROOT = find_project_root()
print("Project root:", ROOT)

Project root: /home/hatch/workspace/youth-unemployment-repo


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3})
FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "processed" / "pakistan_labour_clean.csv")
years = df["year"].values
y = df["youth_unemployment_total"].astype(float).values
series = pd.Series(y, index=years)

SPLIT_YEAR = 2015
train_mask = years <= SPLIT_YEAR
test_years = years[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print(f"Train: {years[train_mask][0]}-{SPLIT_YEAR} (n={len(y_train)}) | Test: {test_years[0]}-{test_years[-1]} (n={len(y_test)})")

Train: 1991-2015 (n=25) | Test: 2016-2025 (n=10)


## Step 1 — Baseline forecasts (naive and drift), one-step-ahead

In [3]:
history = list(y_train)
naive_pred, drift_pred = [], []
drift_rate = (y_train[-1] - y_train[0]) / (len(y_train) - 1)  # average yearly change in train
for actual in y_test:
    naive_pred.append(history[-1])
    drift_pred.append(history[-1] + drift_rate)
    history.append(actual)
naive_pred, drift_pred = np.array(naive_pred), np.array(drift_pred)
print(f"Drift rate from training period: {drift_rate:+.4f} pp/year")

Drift rate from training period: +0.2384 pp/year


## Step 2 — ARIMA (order selected by AIC on training data), rolling one-step-ahead

In [4]:
def select_arima_order(train, max_p=3, max_d=2, max_q=3):
    best_aic, best_order = np.inf, (1, 1, 1)
    for p in range(max_p + 1):
        for d in range(max_d + 1):
            for q in range(max_q + 1):
                if p == 0 and q == 0:
                    continue
                try:
                    fit = ARIMA(train, order=(p, d, q)).fit()
                    if fit.aic < best_aic:
                        best_aic, best_order = fit.aic, (p, d, q)
                except Exception:
                    pass
    return best_order

best_order = select_arima_order(y_train)
print("Selected ARIMA order (by training AIC):", best_order)

history = list(y_train)
arima_pred = []
for actual in y_test:
    fit = ARIMA(history, order=best_order).fit()
    arima_pred.append(fit.forecast(steps=1)[0])
    history.append(actual)
arima_pred = np.array(arima_pred)

/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params =

/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as star

/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using z

/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as star

/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/home/hatch/workspace/youth-unemployment-repo/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using z

Selected ARIMA order (by training AIC): (2, 1, 0)


## Step 3 — Machine-learning models on lag features (one-step-ahead)

In [5]:
def make_supervised(values, lags=(1, 2, 3)):
    rows, targets, yrs = [], [], []
    vals = pd.Series(values)
    roll3 = vals.rolling(3).mean()
    for i in range(max(lags), len(values)):
        rows.append([values[i - l] for l in lags] + [roll3.iloc[i - 1]])
        targets.append(values[i])
        yrs.append(years[i])
    return np.array(rows), np.array(targets), np.array(yrs)

X, y_sup, yrs_sup = make_supervised(y)
train_sup = yrs_sup <= SPLIT_YEAR
X_tr, y_tr = X[train_sup], y_sup[train_sup]
print(f"Supervised rows: train n={train_sup.sum()}, test n={(~train_sup).sum()}")

lr = LinearRegression().fit(X_tr, y_tr)
rf = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_tr, y_tr)

test_mask_sup = ~train_sup
lr_pred = lr.predict(X[test_mask_sup])
rf_pred = rf.predict(X[test_mask_sup])
assert (yrs_sup[test_mask_sup] == test_years).all()
print("Feature order: [lag1, lag2, lag3, rolling_mean_3(prev)]")
print("Linear regression coefficients:", np.round(lr.coef_, 3), "intercept:", round(lr.intercept_, 3))
print("Random forest feature importances:", np.round(rf.feature_importances_, 3))

Supervised rows: train n=22, test n=10


Feature order: [lag1, lag2, lag3, rolling_mean_3(prev)]
Linear regression coefficients: [ 0.682 -0.317  0.528  0.297] intercept: 0.142
Random forest feature importances: [0.243 0.22  0.312 0.225]


## Step 4 — Honest comparison: MAE and RMSE on the 2016–2025 test years

In [6]:
def mae(a, b): return float(np.mean(np.abs(a - b)))
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

preds = {
    "Naive (last value)": naive_pred,
    "Drift": drift_pred,
    f"ARIMA{best_order}": arima_pred,
    "Linear Regression (lags)": lr_pred,
    "Random Forest (lags)": rf_pred,
}
metrics = pd.DataFrame([
    {"model": name, "test_years": f"{test_years[0]}-{test_years[-1]}",
     "n_test": len(y_test), "MAE": mae(y_test, p), "RMSE": rmse(y_test, p)}
    for name, p in preds.items()
]).sort_values("MAE").reset_index(drop=True)

metrics.to_csv(ROOT / "results" / "model_metrics.csv", index=False)
print("Saved results/model_metrics.csv")
metrics

Saved results/model_metrics.csv


,model,test_years,n_test,MAE,RMSE
0,Drift,2016-2025,10,0.886400,1.101069
1,Naive (last value),2016-2025,10,0.920600,1.137701
2,"ARIMA(2, 1, 0)",2016-2025,10,1.184504,1.424247
3,Linear Regression (lags),2016-2025,10,1.741740,2.010957
4,Random Forest (lags),2016-2025,10,3.724507,4.126376


## Step 5 — Plot: actual vs one-step-ahead predictions on the test years

In [7]:
fig, ax = plt.subplots()
ax.plot(test_years, y_test, marker="o", lw=2, color="black", label="Actual")
for name, p in preds.items():
    ax.plot(test_years, p, marker=".", ls="--", label=name)
ax.set_xlabel("Year"); ax.set_ylabel("Youth unemployment (%)")
ax.set_title("One-step-ahead test predictions, 2016–2025")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_test_predictions.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig5_test_predictions.png")

Saved results/figures/fig5_test_predictions.png


## Step 6 — A cautious 5-year outlook, 2026–2030

The best test-MAE model is refitted / rolled forward on the **full** series to produce a
purely indicative 5-year outlook. For ARIMA this uses the model's own multi-step forecast
(with its 95% interval); for the other models the forecasts are generated recursively
(each prediction becomes an input for the next year).

### ⚠️ Uncertainty — read before quoting any of these numbers
- The dataset is **annual with only 35 observations**; five years ahead is a long horizon
  for such a short series, and intervals widen quickly.
- Youth unemployment reacts to shocks no model here can foresee — the 2020–2021 jump in
  the data happened during the COVID-19 pandemic.
- These numbers are a **baseline scenario under 'recent patterns continue'**, not a
  promise and not policy advice.

In [8]:
best_model_name = metrics.loc[0, "model"]
print("Best model on test MAE:", best_model_name)

future_years = np.arange(2026, 2031)
if best_model_name.startswith("ARIMA"):
    fit_full = ARIMA(y, order=best_order).fit()
    fc = fit_full.get_forecast(steps=5)
    point = fc.predicted_mean.values
    lower = fc.conf_int(alpha=0.05).iloc[:, 0].values
    upper = fc.conf_int(alpha=0.05).iloc[:, 1].values
else:
    point, lower, upper = [], [], []
    hist = list(y)
    for step, fy in enumerate(future_years):
        if best_model_name.startswith("Naive"):
            nxt = hist[-1]
        elif best_model_name.startswith("Drift"):
            nxt = hist[-1] + drift_rate
        else:
            feats = np.array([[hist[-1], hist[-2], hist[-3], np.mean(hist[-3:])]])
            model = lr if best_model_name.startswith("Linear") else rf
            nxt = float(model.predict(feats)[0])
        point.append(nxt); lower.append(np.nan); upper.append(np.nan)
        hist.append(nxt)
    point, lower, upper = np.array(point), np.array(lower), np.array(upper)

forecast_df = pd.DataFrame({
    "year": future_years, "forecast": np.round(point, 3),
    "lower_95": np.round(lower, 3), "upper_95": np.round(upper, 3),
    "model": best_model_name,
})
forecast_df.to_csv(ROOT / "results" / "forecast_2026_2030.csv", index=False)
print("Saved results/forecast_2026_2030.csv")
forecast_df

Best model on test MAE: Drift
Saved results/forecast_2026_2030.csv


,year,forecast,lower_95,upper_95,model
0,2026,9.832,NaN,NaN,Drift
1,2027,10.071,NaN,NaN,Drift
2,2028,10.309,NaN,NaN,Drift
3,2029,10.548,NaN,NaN,Drift
4,2030,10.786,NaN,NaN,Drift


In [9]:
fig, ax = plt.subplots()
ax.plot(years, y, marker="o", ms=3, label="Observed (World Bank / ILO modelled)")
ax.plot(future_years, point, marker="s", ls="--", color="#d62728",
        label=f"Outlook 2026–2030 ({best_model_name})")
if not np.isnan(lower).all():
    ax.fill_between(future_years, lower, upper, color="#d62728", alpha=0.15,
                    label="95% interval")
ax.set_xlabel("Year"); ax.set_ylabel("Youth unemployment (%)")
ax.set_title("Pakistan youth unemployment — history and cautious 5-year outlook")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig6_forecast_outlook.png", dpi=150)
plt.close(fig)
print("Saved results/figures/fig6_forecast_outlook.png")
print("\nReminder: the outlook is indicative only — see the uncertainty note above.")

Saved results/figures/fig6_forecast_outlook.png

Reminder: the outlook is indicative only — see the uncertainty note above.
